# TP02 — Cleaning Data with OOP and Design Patterns
**Advanced Programming for Data Science**

This notebook completes Exercises 1–5: CSV reading, missing-value strategies, logging and timing decorators, transformation factory, and a complete cleaning pipeline.

Before running, make sure `sample_data (2)(1).csv` is in the same folder as this notebook.

## Requirements

If pandas is not installed, run this in a code cell:

```python
%pip install pandas
```

The notebook saves the cleaned dataset as `cleaned_data.csv`.

In [ ]:
from abc import ABC, abstractmethod
from functools import wraps
from pathlib import Path
from time import perf_counter
import logging

import pandas as pd


# ---------------------------------------------------------------------------
# Logging configuration
# ---------------------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)


# ---------------------------------------------------------------------------

## Exercise 3: Decorators for logging and timing

In [ ]:
# Exercise 3: Decorators for logging and timing
# ---------------------------------------------------------------------------

def log_action(func):
    """Log when a function starts and finishes."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        logging.info("Starting: %s", func.__qualname__)
        try:
            return func(*args, **kwargs)
        finally:
            logging.info("Finished: %s", func.__qualname__)
    return wrapper


def log_time(func):
    """Measure and log a function's execution time."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            elapsed = perf_counter() - start_time
            logging.info("%s took %.6f seconds", func.__qualname__, elapsed)
    return wrapper


# ---------------------------------------------------------------------------

## Exercise 1: CSVReader class

In [ ]:
# Exercise 1: CSVReader class
# ---------------------------------------------------------------------------

class CSVReader:
    """Read a CSV file and preview its contents."""

    def __init__(self, file_path):
        self._file_path = Path(file_path)
        self._data = None

    @property
    def file_path(self):
        """Return the CSV file path."""
        return self._file_path

    @log_action
    @log_time
    def read(self):
        """Load the CSV file into a pandas DataFrame."""
        if not self._file_path.exists():
            raise FileNotFoundError(
                f"CSV file not found: {self._file_path.resolve()}"
            )

        self._data = pd.read_csv(self._file_path)
        logging.info("Loaded %d rows and %d columns.", *self._data.shape)
        return self._data

    def preview(self, n=5):
        """Print the first n rows of the CSV file."""
        if self._data is None:
            raise RuntimeError("Please call read() before preview().")

        if n < 0:
            raise ValueError("n must be zero or a positive integer.")

        print(f"\nPreview of the first {n} row(s):")
        print(self._data.head(n).to_string(index=False))


# ---------------------------------------------------------------------------

## Exercise 2: Strategy Pattern for missing values

In [ ]:
# Exercise 2: Strategy Pattern for missing values
# ---------------------------------------------------------------------------

class MissingValueStrategy(ABC):
    """Abstract interface for missing-value handling strategies."""

    @abstractmethod
    def handle(self, df):
        """Return a DataFrame after handling missing values."""
        raise NotImplementedError


class DropMissing(MissingValueStrategy):
    """Remove rows that contain at least one missing value."""

    def handle(self, df):
        return df.dropna().copy()


class FillMean(MissingValueStrategy):
    """Fill missing numeric values with their column means.

    Non-numeric columns are left unchanged. This strategy is useful when
    missing values occur in numeric columns and should not remove rows.
    """

    def handle(self, df):
        result = df.copy()
        numeric_columns = result.select_dtypes(include="number").columns
        result[numeric_columns] = result[numeric_columns].fillna(
            result[numeric_columns].mean()
        )
        return result


class FillMode(MissingValueStrategy):
    """Fill missing values with the mode of each column.

    If a column has no available mode, its missing values are left unchanged.
    """

    def handle(self, df):
        result = df.copy()

        for column in result.columns:
            modes = result[column].mode(dropna=True)
            if not modes.empty:
                result[column] = result[column].fillna(modes.iloc[0])

        return result


class DataCleaner:
    """Apply a selected missing-value strategy."""

    def __init__(self, strategy):
        if not isinstance(strategy, MissingValueStrategy):
            raise TypeError("strategy must inherit from MissingValueStrategy.")
        self._strategy = strategy

    @log_action
    @log_time
    def clean(self, df):
        """Apply the selected strategy and report the missing-value count."""
        before = int(df.isna().sum().sum())
        cleaned_df = self._strategy.handle(df)
        after = int(cleaned_df.isna().sum().sum())

        logging.info("Missing values before cleaning: %d", before)
        logging.info("Missing values after cleaning: %d", after)
        return cleaned_df


# ---------------------------------------------------------------------------

## Exercise 4: Factory Pattern for data transformations

In [ ]:
# Exercise 4: Factory Pattern for data transformations
# ---------------------------------------------------------------------------

class DataTransform(ABC):
    """Abstract interface for DataFrame transformations."""

    @abstractmethod
    def apply(self, df):
        """Return a transformed copy of the DataFrame."""
        raise NotImplementedError


class NormalizeColumns(DataTransform):
    """Normalize column names to lowercase snake_case."""

    def apply(self, df):
        result = df.copy()
        normalized_columns = (
            result.columns.astype(str)
            .str.strip()
            .str.lower()
            .str.replace(r"[^a-z0-9]+", "_", regex=True)
            .str.strip("_")
        )
        result.columns = normalized_columns
        return result


class RemoveDuplicates(DataTransform):
    """Remove duplicated rows."""

    def apply(self, df):
        return df.drop_duplicates().reset_index(drop=True)


class StandardizeText(DataTransform):
    """Trim leading/trailing spaces and collapse repeated spaces in text."""

    def apply(self, df):
        result = df.copy()
        text_columns = result.select_dtypes(include=["object", "string"]).columns

        for column in text_columns:
            result[column] = result[column].apply(
                lambda value: " ".join(value.split()) if isinstance(value, str)
                else value
            )

        return result


class TransformFactory:
    """Create transformation objects from a transformation name."""

    _TRANSFORMS = {
        "normalize_columns": NormalizeColumns,
        "remove_duplicates": RemoveDuplicates,
        "standardize_text": StandardizeText,
    }

    @classmethod
    def create_transform(cls, transform_name):
        """Return the requested transformation object."""
        name = transform_name.strip().lower()

        if name not in cls._TRANSFORMS:
            available = ", ".join(cls._TRANSFORMS.keys())
            raise ValueError(
                f"Unknown transformation '{transform_name}'. "
                f"Available transformations: {available}"
            )

        return cls._TRANSFORMS[name]()


# ---------------------------------------------------------------------------

## Exercise 5: Template Method Pattern for the complete pipeline

In [ ]:
# Exercise 5: Template Method Pattern for the complete pipeline
# ---------------------------------------------------------------------------

class DataPipeline(ABC):
    """Define the common steps of a data-cleaning pipeline."""

    def __init__(self, input_path, output_path):
        self.input_path = Path(input_path)
        self.output_path = Path(output_path)
        self.df = None

    def run(self):
        """Execute the pipeline in a fixed order."""
        logging.info("========== Data pipeline started ==========")
        self.load()
        self.clean()
        self.transform()
        self.save()
        logging.info("=========== Data pipeline finished ==========")
        return self.df

    @abstractmethod
    def load(self):
        """Load source data."""
        raise NotImplementedError

    @abstractmethod
    def clean(self):
        """Handle missing values."""
        raise NotImplementedError

    @abstractmethod
    def transform(self):
        """Apply data transformations."""
        raise NotImplementedError

    @abstractmethod
    def save(self):
        """Save the final data."""
        raise NotImplementedError


class CSVDataPipeline(DataPipeline):
    """CSV pipeline that reuses the reader, cleaner, and transform factory."""

    def __init__(
        self,
        input_path,
        output_path,
        missing_strategy="mean",
        transformations=None,
    ):
        super().__init__(input_path, output_path)

        strategies = {
            "drop": DropMissing,
            "mean": FillMean,
            "mode": FillMode,
        }

        strategy_name = missing_strategy.strip().lower()
        if strategy_name not in strategies:
            available = ", ".join(strategies.keys())
            raise ValueError(
                f"Unknown missing-value strategy '{missing_strategy}'. "
                f"Choose from: {available}"
            )

        self.cleaner = DataCleaner(strategies[strategy_name]())
        self.transformations = transformations or [
            "normalize_columns",
            "standardize_text",
            "remove_duplicates",
        ]

    def load(self):
        """Step 1: Load the source CSV file."""
        reader = CSVReader(self.input_path)
        self.df = reader.read()
        reader.preview(5)

    def clean(self):
        """Step 2: Handle missing values with the selected strategy."""
        self.df = self.cleaner.clean(self.df)

    def transform(self):
        """Step 3: Apply each transformation in the configured order."""
        for transform_name in self.transformations:
            transform = TransformFactory.create_transform(transform_name)
            self.df = transform.apply(self.df)
            logging.info("Applied transformation: %s", transform_name)

    def save(self):
        """Step 4: Save the cleaned DataFrame to a new CSV file."""
        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        self.df.to_csv(self.output_path, index=False)
        logging.info("Cleaned data saved to: %s", self.output_path.resolve())
        print("\nCleaned data preview:")
        print(self.df.head().to_string(index=False))


# ---------------------------------------------------------------------------

## Main program

In [ ]:
# Main program
# ---------------------------------------------------------------------------

def main():
    """Run the exercises and the complete cleaning pipeline."""
    project_folder = Path(__file__).resolve().parent
    input_file = project_folder / "sample_data (2)(1).csv"
    output_file = project_folder / "cleaned_data.csv"

## Exercise 1: Read and preview the CSV file.

In [ ]:
# Exercise 1: Read and preview the CSV file.
    reader = CSVReader(input_file)
    original_df = reader.read()
    reader.preview(5)

## Exercise 2: Demonstrate the three missing-value strategies.

In [ ]:
# Exercise 2: Demonstrate the three missing-value strategies.
    print("\n--- Exercise 2: Missing-value strategies ---")
    strategies = {
        "DropMissing": DropMissing(),
        "FillMean": FillMean(),
        "FillMode": FillMode(),
    }

    for strategy_name, strategy in strategies.items():
        result = DataCleaner(strategy).clean(original_df)
        print(
            f"{strategy_name}: {result.shape[0]} rows, "
            f"{int(result.isna().sum().sum())} missing value(s)"
        )

## Exercise 4: Demonstrate the factory.

In [ ]:
# Exercise 4: Demonstrate the factory.
    print("\n--- Exercise 4: Transformation factory ---")
    normalized_df = TransformFactory.create_transform(
        "normalize_columns"
    ).apply(original_df)
    print("Normalized columns:", normalized_df.columns.tolist())

## Exercise 5: Run the full pipeline.

In [ ]:
# Exercise 5: Run the full pipeline.
    print("\n--- Exercise 5: Full cleaning pipeline ---")
    pipeline = CSVDataPipeline(
        input_path=input_file,
        output_path=output_file,
        missing_strategy="mean",
        transformations=[
            "normalize_columns",
            "standardize_text",
            "remove_duplicates",
        ],
    )
    pipeline.run()

## Run the complete pipeline

The following cell runs the pipeline using the provided CSV filename and writes the result to `cleaned_data.csv`.

In [ ]:
def main():
    """Run the exercises and the complete cleaning pipeline."""
    input_file = Path("sample_data (2)(1).csv")
    output_file = Path("cleaned_data.csv")

    # Exercise 1: Read and preview the CSV file.
    reader = CSVReader(input_file)
    original_df = reader.read()
    reader.preview(5)

    # Exercise 2: Demonstrate the three missing-value strategies.
    print("\n--- Exercise 2: Missing-value strategies ---")
    strategies = {
        "DropMissing": DropMissing(),
        "FillMean": FillMean(),
        "FillMode": FillMode(),
    }
    for strategy_name, strategy in strategies.items():
        result = DataCleaner(strategy).clean(original_df)
        print(f"{strategy_name}: {result.shape[0]} rows, "
              f"{int(result.isna().sum().sum())} missing value(s)")

    # Exercise 4: Demonstrate the transformation factory.
    print("\n--- Exercise 4: Transformation factory ---")
    normalized_df = TransformFactory.create_transform("normalize_columns").apply(original_df)
    print("Normalized columns:", normalized_df.columns.tolist())

    # Exercise 5: Run the full cleaning pipeline.
    print("\n--- Exercise 5: Full cleaning pipeline ---")
    pipeline = CSVDataPipeline(
        input_path=input_file,
        output_path=output_file,
        missing_strategy="mean",
        transformations=["normalize_columns", "standardize_text", "remove_duplicates"],
    )
    pipeline.run()

main()